## vllm rollingbatch Qwen3 deployment guide
In this tutorial, you will use LMI container from DLC to SageMaker and run inference with it.

Please make sure the following permission granted before running the notebook:

* SageMaker access

## Step 1: Let's bump up SageMaker and import stuff

In [ ]:
%pip install sagemaker --upgrade  --quiet

In [ ]:
# Import required libraries
import json
import boto3

from sagemaker.core.common_utils import name_from_base
from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant
from sagemaker.core.resources import Model, EndpointConfig, Endpoint

region = boto3.Session().region_name
role = get_execution_role()

## Step 2: Start building SageMaker endpoint
In this step, we will build SageMaker endpoint from scratch

### Getting the container image URI

Check out available images: [Large Model Inference available DLC](https://github.com/aws/deep-learning-containers/blob/master/available_images.md#large-model-inference-containers)

In [ ]:
# Choose a specific version of LMI image directly:
image_uri = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:0.36.0-lmi22.0.0-cu129"

### Create SageMaker model

Checkout more [configuration options](https://docs.djl.ai/docs/serving/serving/docs/lmi/deployment_guide/configurations.html#environment-variable-configurations).

In [ ]:
# Set up environment variables for DJL LMI with vLLM backend
env = {
    "HF_MODEL_ID": "Qwen/Qwen3-8B",
    "SERVING_FAIL_FAST": "true",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
    "OPTION_MAX_MODEL_LEN": "16384",
}

model_name = name_from_base("qwen3")
endpoint_config_name = f"{model_name}-config"
endpoint_name = f"{model_name}-endpoint"

# Use the DJL LMI container
container_definition = ContainerDefinition(
    image=image_uri,
    environment=env
)

# Create model and endpoint
model = Model.create(
    model_name=model_name,
    primary_container=container_definition,
    execution_role_arn=role,
    region=region
)

endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="Primary",
            model_name=model_name,
            instance_type="ml.g7e.2xlarge",
            initial_instance_count=1
        )
    ]
)

### Create SageMaker endpoint

You need to specify the instance to use and endpoint names

In [ ]:
endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name
)

In [ ]:
endpoint.wait_for_status("InService")

### Step 3: Run inference

In [ ]:
request = {"prompt": "The future of Artificial Intelligence is", "max_tokens": 128}
result = endpoint.invoke(
    json.dumps(request),
    content_type="application/json"
)
response = json.loads(result.body.read().decode('utf-8'))
print(f"Response: {response}")

## Clean up the environment

In [ ]:
model.delete()
endpoint.delete()
endpoint_config.delete()